# Task 05: GAN on MNIST

This notebook is built step by step. The goal is to train a small Generative Adversarial Network (GAN) that learns to generate handwritten digit images similar to MNIST.

### 1. Imports and setup

This first cell imports the tools needed later and configures matplotlib in the same style as Task 04.

In [ ]:
from pathlib import Path

from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.training import save_checkpoint, load_checkpoint

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from tqdm import tqdm

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks 
%matplotlib inline  
import matplotlib.pyplot as plt   # THEN import pyplot

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
# get and if needed create figure directory for this task
fig_dir = get_figure_dir("task_05")

# get and if needed create data directory for this task
data_dir = Path("../../../data/task_05")
data_dir.mkdir(parents=True, exist_ok=True)

### 2. Hyperparameters

These values define the size of both GAN variants and the length of training. The discriminator gets a lower learning rate than the generator so it is less likely to dominate immediately.

In [ ]:
learning_rate_generator = 2e-4
learning_rate_discriminator = 1e-4
adam_betas = (0.5, 0.999)

batch_size = 128
num_epochs = 40

# The generator starts from a random vector with this many numbers.
latent_dimension = 128

# MNIST images have shape 1 x 28 x 28.
# The MLP GAN uses flattened image vectors with this length.
image_dimension = 28 * 28 * 1

# Stabilization settings used for both GAN variants.
real_label_min = 0.8
real_label_max = 1.0
fake_label_min = 0.0
fake_label_max = 0.1
instance_noise_start = 0.05
instance_noise_end = 0.0

# The same generated images can be shown every few epochs.
sample_every = 5
num_sample_images = 64

print("latent_dimension:", latent_dimension)
print("image_dimension:", image_dimension)
print("generator learning rate:", learning_rate_generator)
print("discriminator learning rate:", learning_rate_discriminator)

### 3. Dataset and DataLoader

This cell loads the MNIST training images. The transform converts each image to a tensor and normalizes pixel values from `[0, 1]` to approximately `[-1, 1]`. The dataset is stored in the repository-level `data/task_05/` folder, similar to the previous tasks.

In [ ]:
# The generator later uses tanh, so real images are normalized to the same [-1, 1] range.
myTransforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# train=True selects the 60,000 MNIST training images.
# download=True downloads the files only if they are not already present.
dataset = datasets.MNIST(
    root=data_dir,
    train=True,
    transform=myTransforms,
    download=True,
)

# The DataLoader creates mini-batches and shuffles the order each epoch.
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
)

print("Number of training images:", len(dataset))
print("Number of batches per epoch:", len(loader))

### 4. Visualize real MNIST samples

This cell displays a batch of real MNIST images. Since the images were normalized to `[-1, 1]`, `make_grid(..., normalize=True)` rescales them for plotting.

In [ ]:
real_images, real_labels = next(iter(loader))

image_grid = utils.make_grid(
    real_images[:num_sample_images],
    nrow=8,
    normalize=True,
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image_grid.permute(1, 2, 0))
ax.set_title("Real MNIST Samples")
ax.axis("off")

fig.tight_layout()
fig.savefig(fig_dir / "real_mnist_samples.pdf")
None

### 5. MLP GAN model variant

The MLP GAN is the current balanced model. It works with flattened MNIST images of length `784`. Batch normalization helps the generator, while dropout and a smaller discriminator reduce the chance that the discriminator wins immediately.

In [ ]:
class MLPGenerator(nn.Module):
    def __init__(self):
        super().__init__()

        self.gen = nn.Sequential(
            nn.Linear(latent_dimension, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),

            nn.Linear(1024, image_dimension),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.gen(x)


class MLPDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.disc = nn.Sequential(
            nn.Linear(image_dimension, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.disc(x)


mlp_generator = MLPGenerator().to(device)
mlp_discriminator = MLPDiscriminator().to(device)

noise = torch.randn(4, latent_dimension).to(device)
mlp_fake_images = mlp_generator(noise)
mlp_fake_logits = mlp_discriminator(mlp_fake_images)

print("MLP generator output:", mlp_fake_images.shape)
print("MLP discriminator output:", mlp_fake_logits.shape)

### 6. Convolutional GAN model variant

The convolutional GAN keeps the image shape instead of flattening it. Convolutional layers are better suited for images because nearby pixels are processed together, which helps the model learn strokes and local digit structure.

In [ ]:
class ConvGenerator(nn.Module):
    def __init__(self):
        super().__init__()

        self.project = nn.Sequential(
            nn.Linear(latent_dimension, 128 * 7 * 7),
            nn.BatchNorm1d(128 * 7 * 7),
            nn.ReLU(),
        )

        self.conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.project(x)
        x = x.view(-1, 128, 7, 7)
        return self.conv(x)


class ConvDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.disc = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(0.3),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
        )

    def forward(self, x):
        return self.disc(x)


conv_generator = ConvGenerator().to(device)
conv_discriminator = ConvDiscriminator().to(device)

noise = torch.randn(4, latent_dimension).to(device)
conv_fake_images = conv_generator(noise)
conv_fake_logits = conv_discriminator(conv_fake_images)

print("Conv generator output:", conv_fake_images.shape)
print("Conv discriminator output:", conv_fake_logits.shape)

### 7. Loss function, optimizers, and fixed noise

Both GAN variants use `BCEWithLogitsLoss`, separate optimizers, and fixed noise for sample images. Fixed noise makes progress easier to compare because the same latent vectors are used at different epochs.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

mlp_opt_generator = torch.optim.Adam(
    mlp_generator.parameters(),
    lr=learning_rate_generator,
    betas=adam_betas,
)

mlp_opt_discriminator = torch.optim.Adam(
    mlp_discriminator.parameters(),
    lr=learning_rate_discriminator,
    betas=adam_betas,
)

conv_opt_generator = torch.optim.Adam(
    conv_generator.parameters(),
    lr=learning_rate_generator,
    betas=adam_betas,
)

conv_opt_discriminator = torch.optim.Adam(
    conv_discriminator.parameters(),
    lr=learning_rate_discriminator,
    betas=adam_betas,
)

fixed_noise_mlp = torch.randn(num_sample_images, latent_dimension).to(device)
fixed_noise_conv = torch.randn(num_sample_images, latent_dimension).to(device)

mlp_losses_generator = []
mlp_losses_discriminator = []
conv_losses_generator = []
conv_losses_discriminator = []

print(criterion)

### 8. Helper functions for samples, labels, and checkpoints

These helpers are shared by the MLP and convolutional GANs. The sample helper accepts both flattened MLP outputs and image-shaped convolutional outputs.

In [ ]:
def images_for_discriminator(images, model_type):
    if model_type == "mlp":
        return images.view(-1, image_dimension)

    return images


def images_for_plot(images):
    if images.ndim == 2:
        return images.reshape(-1, 1, 28, 28)

    return images


def add_instance_noise(images, noise_std):
    if noise_std <= 0:
        return images

    noisy_images = images + noise_std * torch.randn_like(images)
    return torch.clamp(noisy_images, -1.0, 1.0)


def make_soft_labels(batch_size, label_min, label_max):
    labels = torch.empty(batch_size, 1).uniform_(label_min, label_max)
    return labels.to(device)


def show_generated_samples(generator, noise, title, filename=None):
    generator.eval()

    with torch.no_grad():
        fake = generator(noise)
        fake = images_for_plot(fake)

    image_grid = utils.make_grid(
        fake.cpu(),
        nrow=8,
        normalize=True,
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image_grid.permute(1, 2, 0))
    ax.set_title(title)
    ax.axis("off")

    fig.tight_layout()

    if filename is not None:
        fig.savefig(fig_dir / filename)

    generator.train()
    return fig, ax


def save_gan_checkpoint(
    checkpoint_path,
    model_name,
    model_type,
    generator,
    discriminator,
    opt_generator,
    opt_discriminator,
    epoch,
    losses_generator,
    losses_discriminator,
    fixed_noise,
):
    save_checkpoint(
        path=checkpoint_path,
        model=generator,
        optimizer=opt_generator,
        epoch=epoch,
        train_losses=losses_generator,
        val_losses=[],
        best_val_loss=losses_generator[-1],
        extra={
            "model_name": model_name,
            "model_type": model_type,
            "generator_state_dict": generator.state_dict(),
            "discriminator_state_dict": discriminator.state_dict(),
            "optimizer_generator_state_dict": opt_generator.state_dict(),
            "optimizer_discriminator_state_dict": opt_discriminator.state_dict(),
            "losses_generator": losses_generator,
            "losses_discriminator": losses_discriminator,
            "hyperparameters": {
                "learning_rate_generator": learning_rate_generator,
                "learning_rate_discriminator": learning_rate_discriminator,
                "adam_betas": adam_betas,
                "batch_size": batch_size,
                "num_epochs": num_epochs,
                "latent_dimension": latent_dimension,
                "image_dimension": image_dimension,
                "sample_every": sample_every,
                "num_sample_images": num_sample_images,
                "real_label_min": real_label_min,
                "real_label_max": real_label_max,
                "fake_label_min": fake_label_min,
                "fake_label_max": fake_label_max,
                "instance_noise_start": instance_noise_start,
                "instance_noise_end": instance_noise_end,
            },
            "fixed_noise": fixed_noise.detach().cpu(),
        },
    )


def load_gan_checkpoint(
    checkpoint_path,
    generator,
    discriminator,
    opt_generator,
    opt_discriminator,
):
    checkpoint = load_checkpoint(
        checkpoint_path,
        model=generator,
        optimizer=opt_generator,
        device=device,
    )

    discriminator.load_state_dict(checkpoint["discriminator_state_dict"])
    opt_discriminator.load_state_dict(
        checkpoint["optimizer_discriminator_state_dict"]
    )

    fixed_noise = checkpoint["fixed_noise"].to(device)

    return (
        checkpoint,
        checkpoint["train_losses"],
        checkpoint["losses_discriminator"],
        fixed_noise,
    )

### 9. Checkpoint settings and untrained samples

Each model variant has its own skip flag and checkpoint. The epoch-0 sample images are saved before training so the trained samples can be compared against the random starting point.

In [ ]:
checkpoint_path_mlp = Path("../models/mnist_gan_mlp.pt")
checkpoint_path_conv = Path("../models/mnist_gan_conv.pt")

show_generated_samples(
    mlp_generator,
    fixed_noise_mlp,
    title="MLP GAN Samples Before Training",
    filename="mlp_generated_epoch_00.pdf",
)

show_generated_samples(
    conv_generator,
    fixed_noise_conv,
    title="Conv GAN Samples Before Training",
    filename="conv_generated_epoch_00.pdf",
)

None

### 10. Shared GAN training loop

The GAN loop is custom because it trains two models. The discriminator update and generator update are kept separate. Both variants use label smoothing, a lower discriminator learning rate, and small input noise for the discriminator that decays over epochs.

In [ ]:
def train_gan(
    model_name,
    model_type,
    generator,
    discriminator,
    opt_generator,
    opt_discriminator,
    checkpoint_path,
    fixed_noise,
    skip_training,
):
    if skip_training:
        checkpoint, losses_generator, losses_discriminator, fixed_noise = (
            load_gan_checkpoint(
                checkpoint_path,
                generator,
                discriminator,
                opt_generator,
                opt_discriminator,
            )
        )

        print("Loaded checkpoint from:", checkpoint_path)
        print("Completed epochs:", checkpoint["epoch"])

        show_generated_samples(
            generator,
            fixed_noise,
            title=f"{model_name.upper()} GAN Samples From Epoch {checkpoint['epoch']}",
            filename=f"{model_name}_generated_epoch_{checkpoint['epoch']:02d}_loaded.pdf",
        )
        plt.show()

        return losses_generator, losses_discriminator, fixed_noise

    losses_generator = []
    losses_discriminator = []

    for epoch in range(num_epochs):
        epoch_loss_discriminator = 0.0
        epoch_loss_generator = 0.0

        progress_bar = tqdm(loader, desc=f"{model_name.upper()} Epoch {epoch + 1}/{num_epochs}")

        epoch_fraction = epoch / max(num_epochs - 1, 1)
        instance_noise_std = (
            instance_noise_start
            + epoch_fraction * (instance_noise_end - instance_noise_start)
        )

        for real, _ in progress_bar:
            real = real.to(device)
            current_batch_size = real.shape[0]

            real_labels = make_soft_labels(
                current_batch_size,
                real_label_min,
                real_label_max,
            )
            fake_labels = make_soft_labels(
                current_batch_size,
                fake_label_min,
                fake_label_max,
            )

            # -----------------------------
            # 1) Train the discriminator
            # -----------------------------
            noise = torch.randn(current_batch_size, latent_dimension).to(device)
            fake = generator(noise)

            real_for_discriminator = add_instance_noise(real, instance_noise_std)
            fake_for_discriminator = add_instance_noise(fake.detach(), instance_noise_std)

            real_for_discriminator = images_for_discriminator(
                real_for_discriminator,
                model_type,
            )
            fake_for_discriminator = images_for_discriminator(
                fake_for_discriminator,
                model_type,
            )

            discriminator_real = discriminator(real_for_discriminator)
            loss_discriminator_real = criterion(discriminator_real, real_labels)

            discriminator_fake = discriminator(fake_for_discriminator)
            loss_discriminator_fake = criterion(discriminator_fake, fake_labels)

            loss_discriminator = (
                loss_discriminator_real + loss_discriminator_fake
            ) / 2

            opt_discriminator.zero_grad()
            loss_discriminator.backward()
            opt_discriminator.step()

            # -----------------------------
            # 2) Train the generator
            # -----------------------------
            noise = torch.randn(current_batch_size, latent_dimension).to(device)
            fake = generator(noise)
            fake_for_generator = add_instance_noise(fake, instance_noise_std)
            fake_for_generator = images_for_discriminator(
                fake_for_generator,
                model_type,
            )

            discriminator_fake = discriminator(fake_for_generator)
            generator_target_labels = make_soft_labels(
                current_batch_size,
                real_label_min,
                real_label_max,
            )
            loss_generator = criterion(
                discriminator_fake,
                generator_target_labels,
            )

            opt_generator.zero_grad()
            loss_generator.backward()
            opt_generator.step()

            epoch_loss_discriminator += loss_discriminator.item()
            epoch_loss_generator += loss_generator.item()

            progress_bar.set_postfix(
                loss_D=f"{loss_discriminator.item():.4f}",
                loss_G=f"{loss_generator.item():.4f}",
                noise=f"{instance_noise_std:.3f}",
            )

        epoch_loss_discriminator /= len(loader)
        epoch_loss_generator /= len(loader)

        losses_discriminator.append(epoch_loss_discriminator)
        losses_generator.append(epoch_loss_generator)

        print(
            f"{model_name.upper()} Epoch {epoch + 1:02d}/{num_epochs} | "
            f"Loss discriminator: {epoch_loss_discriminator:.4f} | "
            f"Loss generator: {epoch_loss_generator:.4f} | "
            f"Instance noise: {instance_noise_std:.3f}"
        )

        save_gan_checkpoint(
            checkpoint_path=checkpoint_path,
            model_name=model_name,
            model_type=model_type,
            generator=generator,
            discriminator=discriminator,
            opt_generator=opt_generator,
            opt_discriminator=opt_discriminator,
            epoch=epoch + 1,
            losses_generator=losses_generator,
            losses_discriminator=losses_discriminator,
            fixed_noise=fixed_noise,
        )

        should_save_samples = (
            (epoch + 1) == 1
            or (epoch + 1) % sample_every == 0
            or (epoch + 1) == num_epochs
        )

        if should_save_samples:
            show_generated_samples(
                generator,
                fixed_noise,
                title=f"{model_name.upper()} GAN Samples After Epoch {epoch + 1}",
                filename=f"{model_name}_generated_epoch_{epoch + 1:02d}.pdf",
            )
            plt.show()

    return losses_generator, losses_discriminator, fixed_noise

### 11. Train or load both GAN variants

Run this cell to train both models. After training once, set `skip_training_mlp` or `skip_training_conv` to `True` to load the saved checkpoint instead.

In [ ]:
skip_training_mlp = True
skip_training_conv = True

mlp_losses_generator, mlp_losses_discriminator, fixed_noise_mlp = train_gan(
    model_name="mlp",
    model_type="mlp",
    generator=mlp_generator,
    discriminator=mlp_discriminator,
    opt_generator=mlp_opt_generator,
    opt_discriminator=mlp_opt_discriminator,
    checkpoint_path=checkpoint_path_mlp,
    fixed_noise=fixed_noise_mlp,
    skip_training=skip_training_mlp,
)

conv_losses_generator, conv_losses_discriminator, fixed_noise_conv = train_gan(
    model_name="conv",
    model_type="conv",
    generator=conv_generator,
    discriminator=conv_discriminator,
    opt_generator=conv_opt_generator,
    opt_discriminator=conv_opt_discriminator,
    checkpoint_path=checkpoint_path_conv,
    fixed_noise=fixed_noise_conv,
    skip_training=skip_training_conv,
)

### 12. Compare training losses

This plot compares generator and discriminator losses for both model variants. A discriminator loss that collapses to zero while the generator loss explodes usually means the discriminator is still too strong.

In [ ]:
if (
    len(mlp_losses_generator) == 0
    or len(conv_losses_generator) == 0
):
    print("Losses are missing. Train or load both GAN variants first.")
else:
    mlp_epochs = np.arange(1, len(mlp_losses_generator) + 1)
    conv_epochs = np.arange(1, len(conv_losses_generator) + 1)

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.plot(
        mlp_epochs,
        mlp_losses_discriminator,
        label="MLP discriminator",
        color="C0",
    )
    ax.plot(
        mlp_epochs,
        mlp_losses_generator,
        label="MLP generator",
        color="C1",
    )
    ax.plot(
        conv_epochs,
        conv_losses_discriminator,
        label="Conv discriminator",
        color="C2",
    )
    ax.plot(
        conv_epochs,
        conv_losses_generator,
        label="Conv generator",
        color="C3",
    )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Binary cross entropy loss")
    ax.set_title("GAN Loss Comparison")
    ax.legend()

    fig.tight_layout()
    fig.savefig(fig_dir / "gan_loss_comparison.pdf")

None

### 13. Final generated samples

These final grids use new random noise, not the fixed-noise vectors used during training. The fixed-noise images show progress over epochs; these images show fresh samples from each trained generator.

In [ ]:
final_noise_mlp = torch.randn(num_sample_images, latent_dimension).to(device)
final_noise_conv = torch.randn(num_sample_images, latent_dimension).to(device)

show_generated_samples(
    mlp_generator,
    final_noise_mlp,
    title="MLP GAN Final Generated Samples",
    filename="mlp_final_generated_samples.pdf",
)

show_generated_samples(
    conv_generator,
    final_noise_conv,
    title="Conv GAN Final Generated Samples",
    filename="conv_final_generated_samples.pdf",
)

None

### Conclusion

The MLP GAN is a useful baseline because it shows the adversarial training idea with simple linear layers. The convolutional GAN should usually perform better on MNIST because convolutional filters learn local stroke patterns while preserving the image structure.

A better model should show clearer digit strokes, more digit diversity, and less speckle or random noise. The loss curves should also stay in a reasonable range: discriminator loss should not collapse to zero immediately, and generator loss should not explode.

---

## Bonus Task: Wasserstein GAN with Gradient Penalty

The bonus task uses a Wasserstein GAN (WGAN). The main idea is to replace the discriminator with a critic. A discriminator predicts a probability for real or fake images, while a critic gives a real-valued score. Higher scores should be assigned to real images, and lower scores should be assigned to generated images.

The first WGAN version used weight clipping, which can make the critic too restricted and unstable. This version uses WGAN-GP, which is a standard WGAN variant. The gradient penalty keeps the critic smooth without forcing all weights into a tiny interval.

This section is split from the previous MLP and convolutional GAN comparison, but it reuses the same MNIST data, plotting helpers, checkpoint utilities, and comparison idea.

### Bonus 1. WGAN-GP setup

This cell defines the WGAN-GP-specific settings. The checkpoint and generated sample names use the `wgan_gp_` prefix so the previous MLP, convolutional GAN, and old WGAN outputs are kept separate.

In [ ]:
wgan_name = "wgan_gp_k3"

# After training once, set this to True to load the WGAN-GP checkpoint.
skip_training_wgan = True
checkpoint_path_wgan = Path("../models/mnist_wgan_gp_k3.pt")

wgan_num_epochs = 60

# WGAN-GP still trains the critic more often than the generator.
critic_updates_per_generator_update = 3

# Set this to None for full training. The value 100 is useful for quick tests.
max_batches_per_epoch_wgan = None

# WGAN-GP commonly uses Adam with these beta values.
wgan_learning_rate = 1e-4
wgan_adam_betas = (0.0, 0.9)
lambda_gp = 10

fixed_noise_wgan = torch.randn(num_sample_images, latent_dimension).to(device)

wgan_losses_critic = []
wgan_losses_generator = []

print("WGAN-GP checkpoint:", checkpoint_path_wgan)
print("Critic updates per generator update:", critic_updates_per_generator_update)
print("Max batches per epoch:", max_batches_per_epoch_wgan)

### Bonus 2. WGAN-GP generator and critic

The WGAN-GP generator creates image tensors with shape `[batch, 1, 28, 28]`. The model is a little deeper than before and uses `3 x 3` convolution layers. This is a common image-modeling choice because small kernels can build up more complex features across multiple layers while keeping the number of parameters manageable.

In [ ]:
class WGANGenerator(nn.Module):
    def __init__(self):
        super().__init__()

        self.project = nn.Sequential(
            nn.Linear(latent_dimension, 128 * 7 * 7),
            nn.BatchNorm1d(128 * 7 * 7),
            nn.ReLU(),
        )

        self.conv = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.project(x)
        x = x.view(-1, 128, 7, 7)
        return self.conv(x)


class WGANCritic(nn.Module):
    def __init__(self):
        super().__init__()

        self.critic = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
        )

    def forward(self, x):
        return self.critic(x)


wgan_generator = WGANGenerator().to(device)
wgan_critic = WGANCritic().to(device)

noise = torch.randn(4, latent_dimension).to(device)
wgan_fake_images = wgan_generator(noise)
wgan_fake_scores = wgan_critic(wgan_fake_images)

print("WGAN generator output:", wgan_fake_images.shape)
print("WGAN critic output:", wgan_fake_scores.shape)

### Bonus 3. WGAN-GP optimizers, gradient penalty, and initial samples

WGAN-GP uses Adam instead of RMSprop. The gradient penalty replaces weight clipping by encouraging the critic gradient norm to stay close to `1` on images interpolated between real and generated images. This is the main stabilization idea in this bonus section.

In [ ]:
wgan_opt_generator = torch.optim.Adam(
    wgan_generator.parameters(),
    lr=wgan_learning_rate,
    betas=wgan_adam_betas,
)

wgan_opt_critic = torch.optim.Adam(
    wgan_critic.parameters(),
    lr=wgan_learning_rate,
    betas=wgan_adam_betas,
)


def calculate_gradient_penalty(critic, real, fake):
    batch_size = real.shape[0]

    epsilon = torch.rand(batch_size, 1, 1, 1).to(device)
    interpolated = epsilon * real + (1 - epsilon) * fake.detach()
    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated)

    gradients = torch.autograd.grad(
        outputs=mixed_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    gradient_penalty = ((gradient_norm - 1) ** 2).mean()

    return gradient_penalty


show_generated_samples(
    wgan_generator,
    fixed_noise_wgan,
    title="WGAN-GP Samples Before Training",
    filename="wgan_gp_generated_epoch_00.pdf",
)

print(wgan_opt_generator)
print(wgan_opt_critic)
None

### Bonus 4. Train or load the WGAN-GP

The critic is trained to give real images higher scores than fake images. The generator is trained to increase the critic score of generated images. The gradient penalty keeps the critic smooth without weight clipping, which should reduce stripe-like collapse and unstable critic dynamics.

In [ ]:
if skip_training_wgan:
    checkpoint_wgan = load_checkpoint(
        checkpoint_path_wgan,
        model=wgan_generator,
        optimizer=wgan_opt_generator,
        device=device,
    )

    wgan_critic.load_state_dict(checkpoint_wgan["critic_state_dict"])
    wgan_opt_critic.load_state_dict(checkpoint_wgan["optimizer_critic_state_dict"])

    wgan_losses_generator = checkpoint_wgan["train_losses"]
    wgan_losses_critic = checkpoint_wgan["losses_critic"]
    fixed_noise_wgan = checkpoint_wgan["fixed_noise"].to(device)

    print("Loaded WGAN-GP checkpoint from:", checkpoint_path_wgan)
    print("Completed epochs:", checkpoint_wgan["epoch"])

    show_generated_samples(
        wgan_generator,
        fixed_noise_wgan,
        title=f"WGAN-GP Samples From Epoch {checkpoint_wgan['epoch']}",
        filename=f"wgan_gp_generated_epoch_{checkpoint_wgan['epoch']:02d}_loaded.pdf",
    )
    plt.show()

else:
    for epoch in range(wgan_num_epochs):
        epoch_loss_critic = 0.0
        epoch_loss_generator = 0.0
        critic_update_count = 0
        generator_update_count = 0
        last_loss_generator = None

        progress_bar = tqdm(loader, desc=f"WGAN-GP Epoch {epoch + 1}/{wgan_num_epochs}")

        for batch_idx, (real, _) in enumerate(progress_bar):
            if (
                max_batches_per_epoch_wgan is not None
                and batch_idx >= max_batches_per_epoch_wgan
            ):
                break

            real = real.to(device)
            current_batch_size = real.shape[0]

            # -----------------------------
            # 1) Train the critic
            # -----------------------------
            noise = torch.randn(current_batch_size, latent_dimension).to(device)
            fake = wgan_generator(noise)

            real_score = wgan_critic(real)
            fake_score = wgan_critic(fake.detach())
            gradient_penalty = calculate_gradient_penalty(
                wgan_critic,
                real,
                fake,
            )

            loss_critic = (
                fake_score.mean()
                - real_score.mean()
                + lambda_gp * gradient_penalty
            )

            wgan_opt_critic.zero_grad()
            loss_critic.backward()
            wgan_opt_critic.step()

            epoch_loss_critic += loss_critic.item()
            critic_update_count += 1

            # -----------------------------
            # 2) Train the generator
            # -----------------------------
            if (batch_idx + 1) % critic_updates_per_generator_update == 0:
                noise = torch.randn(current_batch_size, latent_dimension).to(device)
                fake = wgan_generator(noise)
                fake_score = wgan_critic(fake)

                loss_generator = -fake_score.mean()

                wgan_opt_generator.zero_grad()
                loss_generator.backward()
                wgan_opt_generator.step()

                last_loss_generator = loss_generator.item()
                epoch_loss_generator += last_loss_generator
                generator_update_count += 1

            loss_generator_display = (
                f"{last_loss_generator:.4f}"
                if last_loss_generator is not None
                else "n/a"
            )

            progress_bar.set_postfix(
                loss_C=f"{loss_critic.item():.4f}",
                loss_G=loss_generator_display,
                gp=f"{gradient_penalty.item():.4f}",
            )

        epoch_loss_critic /= critic_update_count
        epoch_loss_generator /= max(generator_update_count, 1)

        wgan_losses_critic.append(epoch_loss_critic)
        wgan_losses_generator.append(epoch_loss_generator)

        print(
            f"WGAN-GP Epoch {epoch + 1:02d}/{wgan_num_epochs} | "
            f"Loss critic: {epoch_loss_critic:.4f} | "
            f"Loss generator: {epoch_loss_generator:.4f}"
        )

        save_checkpoint(
            path=checkpoint_path_wgan,
            model=wgan_generator,
            optimizer=wgan_opt_generator,
            epoch=epoch + 1,
            train_losses=wgan_losses_generator,
            val_losses=[],
            best_val_loss=wgan_losses_generator[-1],
            extra={
                "model_name": wgan_name,
                "model_type": "wgan_gp",
                "generator_state_dict": wgan_generator.state_dict(),
                "critic_state_dict": wgan_critic.state_dict(),
                "optimizer_generator_state_dict": wgan_opt_generator.state_dict(),
                "optimizer_critic_state_dict": wgan_opt_critic.state_dict(),
                "losses_generator": wgan_losses_generator,
                "losses_critic": wgan_losses_critic,
                "hyperparameters": {
                    "wgan_num_epochs": wgan_num_epochs,
                    "critic_updates_per_generator_update": critic_updates_per_generator_update,
                    "max_batches_per_epoch_wgan": max_batches_per_epoch_wgan,
                    "wgan_learning_rate": wgan_learning_rate,
                    "wgan_adam_betas": wgan_adam_betas,
                    "lambda_gp": lambda_gp,
                    "batch_size": batch_size,
                    "latent_dimension": latent_dimension,
                    "num_sample_images": num_sample_images,
                },
                "fixed_noise": fixed_noise_wgan.detach().cpu(),
            },
        )

        if (
            (epoch + 1) == 1
            or (epoch + 1) % sample_every == 0
            or (epoch + 1) == wgan_num_epochs
        ):
            show_generated_samples(
                wgan_generator,
                fixed_noise_wgan,
                title=f"WGAN-GP Samples After Epoch {epoch + 1}",
                filename=f"wgan_gp_generated_epoch_{epoch + 1:02d}.pdf",
            )
            plt.show()

### Bonus 5. Compare WGAN-GP losses with the top GAN results

This cell plots the WGAN-GP losses together with the MLP and convolutional GAN losses from the main part of the notebook above. WGAN-GP losses are not directly comparable to BCE GAN losses, so the plot is mainly useful for checking whether training is stable and whether the generator loss avoids extreme swings.

In [ ]:
if len(wgan_losses_generator) == 0 or len(wgan_losses_critic) == 0:
    print("No WGAN-GP losses available yet. Train or load the WGAN-GP first.")
else:
    wgan_epochs = np.arange(1, len(wgan_losses_generator) + 1)

    fig, ax = plt.subplots(figsize=(7, 4))

    # These curves come from the WGAN-GP bonus section.
    ax.plot(
        wgan_epochs,
        wgan_losses_critic,
        label="Bonus WGAN-GP critic loss",
        color="C4",
    )
    ax.plot(
        wgan_epochs,
        wgan_losses_generator,
        label="Bonus WGAN-GP generator loss",
        color="C5",
    )

    # These curves come from the main GAN comparison above.
    if len(mlp_losses_generator) > 0:
        mlp_epochs = np.arange(1, len(mlp_losses_generator) + 1)
        ax.plot(
            mlp_epochs,
            mlp_losses_generator,
            label="Top MLP GAN generator loss",
            color="C1",
            linestyle=":",
        )

    if len(conv_losses_generator) > 0:
        conv_epochs = np.arange(1, len(conv_losses_generator) + 1)
        ax.plot(
            conv_epochs,
            conv_losses_generator,
            label="Top Conv GAN generator loss",
            color="C3",
            linestyle="--",
        )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Top GAN Results vs Bonus WGAN-GP Losses")
    ax.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(fig_dir / "wgan_gp_vs_top_gan_loss_comparison.pdf")

None

### Bonus 6. Final WGAN-GP samples

This cell creates final WGAN-GP samples from new random noise. The visual check is the same as before: clearer digit strokes, diverse digit shapes, and less noisy speckle are signs of improvement.

In [ ]:
final_noise_wgan = torch.randn(num_sample_images, latent_dimension).to(device)

show_generated_samples(
    wgan_generator,
    final_noise_wgan,
    title="WGAN-GP Final Generated Samples",
    filename="wgan_gp_final_generated_samples.pdf",
)

None

### Bonus 7. Compare generated samples with the top GAN results

This cell shows fresh generated samples from the two GANs trained in the main part of the notebook and from the bonus WGAN-GP. The labels and filenames make it clear which grids come from the top section and which grid comes from the bonus section.

In [ ]:
comparison_noise = torch.randn(num_sample_images, latent_dimension).to(device)

show_generated_samples(
    mlp_generator,
    comparison_noise,
    title="Top MLP GAN Generated Samples",
    filename="top_mlp_generated_samples_for_wgan_gp_comparison.pdf",
)

show_generated_samples(
    conv_generator,
    comparison_noise,
    title="Top Conv GAN Generated Samples",
    filename="top_conv_generated_samples_for_wgan_gp_comparison.pdf",
)

show_generated_samples(
    wgan_generator,
    comparison_noise,
    title="Bonus WGAN-GP Generated Samples",
    filename="bonus_wgan_gp_generated_samples_for_comparison.pdf",
)

None